In [ ]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

In [ ]:
import os
from time import sleep
from pathlib import Path
from urllib.parse import quote, unquote, urlparse, parse_qs
import numpy as np
from openpyxl import load_workbook

from birddog.wiki import (
    get_title,
    WIKI_NAMESPACE,
    )

from birddog.nocodb import (
    NOCODB_API_TOKEN,
    PROJECT_SLUG,
    NOCODB_URL,
    get_table_id,
    upsert_table_data,
    )

In [ ]:
root = "./var/WAT"

In [ ]:
def list_files(directory: str, suffix: str, prefix=None) -> list[Path]:
    """Recursively list only files in `directory` with the given `suffix`,
    skipping any whose name starts with '~'.
    """
    return [
        p for p in Path(directory).rglob(f"*{suffix}")
        if p.is_file() and not p.name.startswith("~")
        if not prefix or p.name.startswith(prefix)
    ]

In [ ]:
archives = list_files(root, "xlsx")
archives

In [ ]:
def find_table_header_row(ws):
    for row in ws.rows:
        for c in row:
            if c.value == "TOTALS:":
                return ws[c.row + 1]
    return None

In [ ]:
def find_formula_row_index(ws):
    for row in ws.rows:
        for c in row:
            if c.value == "TOTALS:":
                return c.row
    return None

In [ ]:
def max_content_col(row):
    for cell in reversed(row):
        if cell.value:
            return cell.column_letter
    return row[0].column_letter

In [ ]:
def partition_sheet(ws):
    result = {}
    f_row = find_formula_row_index(ws)
    if f_row is not None:
        h_row = f_row + 1
        result["table_header"] = f"A{h_row}:{max_content_col(ws[h_row])}{h_row}"
        max_col = "A"
        for row in range(1, f_row-1):
            max_col = max(max_col, max_content_col(ws[row]))
        result["sheet_header"] = f"A1:{max_col}{f_row-1}"
        max_col = "A"
        for row in range((h_row+1), ws.max_row):
            max_col = max(max_col, max_content_col(ws[row]))
            if not ws[row][0].value:
                result["table_content"] = f"A{h_row+1}:{max_col}{row-1}"
                break
    return result

In [ ]:
def extract_strings_in_range(ws, cell_range):
    result = dict()
    for row in ws[cell_range]:
        for cell in row:
            value = cell.value
            if value is not None:
                if isinstance(value, float):
                    value = int(value)
                if not isinstance(value, str):
                    value = str(value)
                if value:
                    result[value] = result.get(value, 0) + 1
    return result

In [ ]:
def extract_strings_by_partition(ws):
    return { key: extract_strings_in_range(ws, cell_range) 
             for key, cell_range in partition_sheet(ws).items() }

In [ ]:
def all_keys(d1, d2):
    return set(d1.keys()).union(d2.keys())

In [ ]:
def merge_hist(hist1, hist2):
    return { key: hist1.get(key, 0) + hist2.get(key, 0)
        for key in all_keys(hist1, hist2) }

In [ ]:
def ranked_hist(hist):
    return sorted(list(hist.items()), key=lambda x: -x[1])

In [ ]:
def truncate_hist(hist, thresh=0, percentile=None):
    hist = ranked_hist(hist)
    if percentile is not None:
        thresh = sum([x[1] for x in hist]) * percentile
    return list(filter(lambda x: x[1] >= thresh, hist))

In [ ]:
def merge_all_hists(a, b):
    return { 
        key: merge_hist(a.get(key, dict()), b.get(key, dict())) 
        for key in all_keys(a, b) }

In [ ]:
def reduce_archive(archives, map_fn, reduce_fn, init=None):
    result = init
    for i, archive in enumerate(archives):
        print(i, archive)
        wb = load_workbook(archive)
        result = reduce_fn(map_fn(wb), result)
    return result    

In [ ]:
archive_strings = reduce_archive(archives[:5],
                         lambda a: extract_strings_by_partition(a.worksheets[0]),
                         lambda a, b: { 
                            key: merge_hist(a.get(key, {}), b.get(key, {})) for key in all_keys(a, b) 
                            },
                         dict())

In [ ]:
ranked_hist(strings["archive_strings"])

In [ ]:
def archive_mapper(archive):
    result = { "archive": extract_strings_by_partition(archive.worksheets[0]) }
    for sheet in archive.worksheets[1:]:
        strings = extract_strings_by_partition(sheet)
        subset = "fund" if sheet.title.startswith("fund") else "opus"
        result[subset] = merge_all_hists(strings, result.get(subset, dict()))
    return result

def archive_reducer(a, b):
    return { 
        key: merge_all_hists(a.get(key, dict()), b.get(key, dict()))
        for key in all_keys(a, b) }

In [ ]:
result = reduce_archive(archives[:5], archive_mapper, archive_reducer, dict())

In [ ]:
truncate_hist(result['fund']['table_header'], 10)

In [ ]:
truncate_hist(result['opus']['sheet_header'], 200)

In [ ]:
truncate_hist(result['archive']['table_content'], percentile=.05)

In [ ]:
for subset in ['archive', 'fund', 'opus']:
    hists = result[subset]
    for key in hists.keys():
        print(f"======= {subset}:{key}: ========")
        print(truncate_hist(result[subset][key], percentile=.05))

In [ ]:
wb = load_workbook(list_files(root, "xlsx", prefix="DAHEO-D-wiki")[0])

In [ ]:
wb.worksheets[0]

In [ ]:
ws=wb.worksheets[0]

In [ ]:
a7 = ws["A7"]

In [ ]:
a7.hyperlink

In [ ]:
a7.value

In [ ]:
get_title(unquote(a7.hyperlink.target), include_namespace=False)

In [ ]:
def get_page_title_from_link(cell):
    if not cell.hyperlink:
        return ""
    url = cell.hyperlink.target
    if "index.php" in url:
        query = urlparse(url).query
        params = parse_qs(query)
        if "title" in params:
            result = params["title"][0]
            ns_prefix = f"{WIKI_NAMESPACE}:"
            if result.startswith(ns_prefix):
                result = result[len(ns_prefix):]
            return result
    return get_title(unquote(url), include_namespace=False)

In [ ]:
def get_cell_value(cell):
    value = cell.value
    if isinstance(value, float):
        value = int(value)
    if not isinstance(value, str):
        value = str(value)
    return value

In [ ]:
def add_page(page: dict, page_table: dict) -> None:
    """Merge a page entry into the page_table by title."""
    title = page["title"]
    # ensure an entry exists for this title
    entry = page_table.setdefault(title, {})
    # update/merge keys
    entry.update(page)

In [ ]:
def process_archive_sheet(ws, page_table={}):
    parent_title = get_page_title_from_link(ws["D3"])
    source_type = get_cell_value(ws["C1"])
    add_page({
        "title": parent_title,
        "label": get_cell_value(ws["A1"]).replace(" ", "-"),
        "level": "archive",
        "description": get_cell_value(ws["A2"]),
        "availability": "linked",
        "change_date": get_cell_value(ws["L1"]),
        "reference_date": get_cell_value(ws["O1"]),
        "doc_links": get_cell_value(ws["B4"]),
        "source_type": source_type,
        "parent": "",
        }, page_table)
    
    for r in range(7, ws.max_row+1):
        cell = ws[f"A{r}"]
        if str(cell.value).startswith("="):
            break
        if cell.value:
            title = get_page_title_from_link(cell)
            label = get_cell_value(cell)
            add_page({
                "title": title,
                "label": label,
                "level": "fond",
                "description": get_cell_value(ws[f"B{r}"]),
                "years": get_cell_value(ws[f"C{r}"]),
                "availability": get_cell_value(ws[f"D{r}"]),
                "source_type": source_type,
                "parent": parent_title,
                }, page_table)
            #print(f"{title}, {description}, {date_range}, {availability}, {doc_link}")
    return page_table

In [ ]:
pages = process_archive_sheet(ws)

In [ ]:
pages['ДАХеО/Д']

In [ ]:
len(pages)

In [ ]:
BASE_ID = "pljzqjmv8a5nvku"
NOCODB_V3_API_ROOT = "https://app.nocodb.com/api/v3"
NOCODB_V2_API_ROOT = "https://app.nocodb.com/api/v2"
TABLE_ID = {
    "pages": "mtrj7h4scl3s15b",
    "documents": "m676bm10pfq3hwk",
}
FIELD_ID = {
    "doc_links": "c42iq5cns4oof05",
    "parent": "coy5kak8p66e1kv",
    "children": "cuj7mxq51rot9kt",
    "owning_page": "cbrvqbad1g0mdiv",
}

In [ ]:
import requests
session = requests.Session()
session.headers.update({
    "xc-token": NOCODB_API_TOKEN,
    "Content-Type": "application/json",
})


In [ ]:
def get_table_id(table_name):
    return TABLE_ID[table_name.lower()]

In [ ]:
def get_field_id(field_name):
    return FIELD_ID[field_name.lower()]

In [ ]:
get_table_id("Pages")

In [ ]:
def list_records(table_name, base_id=BASE_ID):
    #response = session.get(f"{NOCODB_V3_API_ROOT}/data/{base_id}/{table_id}/records")
    response = session.get(f"{NOCODB_V2_API_ROOT}/tables/{get_table_id(table_name)}/records")
    response.raise_for_status()
    return response.json()

In [ ]:
def list_links(table_name, field_name, record_id, base_id=BASE_ID):
    #response = session.get(f"{NOCODB_V3_API_ROOT}/data/{base_id}/{table_id}/records")
    url = f"{NOCODB_V2_API_ROOT}/tables/{get_table_id(table_name)}"
    url += f"/links/{get_field_id(field_name)}"
    url += f"/records/{record_id}"
    response = session.get(url)
    response.raise_for_status()
    return response.json()

In [ ]:
list_records("pages")

In [ ]:
list_links("pages", "children", 1)['list']

In [ ]:
def create_record(table_name, payload, base_id=BASE_ID):
    url = f"{NOCODB_V2_API_ROOT}/tables/{get_table_id(table_name)}/records"
    response = session.post(url, json=payload)
    response.raise_for_status()
    return response.json()

In [ ]:
create_record("pages", { "title": "Test Page 4" })

In [ ]:
def update_record(table_name, payload, base_id=BASE_ID):
    url = f"{NOCODB_V2_API_ROOT}/tables/{get_table_id(table_name)}/records"
    response = session.patch(url, json=payload)
    response.raise_for_status()
    return response.json()

In [ ]:
update_record("pages", { "Id": 5, "comments": "comment update", "years": "1960-1968" })

In [ ]:
create_record("pages", [{ "title": f"page {i}" } for i in range(10,15)])

In [ ]:
update_record("pages", [{ "title": f"page {i}" } for i in range(20,25)])

In [ ]:
def lookup_record_id(table_name, field_name, value, base_id=BASE_ID):
    url = f"{NOCODB_V2_API_ROOT}/tables/{get_table_id(table_name)}/records"
    params = {"fields": "Id", "where": f"({field_name},eq,{value})"}
    print(f"record lookup: {field_name}='{value}'")
    response = session.get(url, params=params)
    response.raise_for_status()
    result = response.json()['list']
    return result[0]['Id'] if result else None

In [ ]:
lookup_record_id("pages", "title", "foo")

In [ ]:
create_record("pages", list(pages.values())[:10])

In [ ]:
list(pages.values())[:10]

In [ ]:
from datetime import datetime

def _iso_date_or_none(s):
    """Accepts '21 Aug 2025', '1905-1912', etc.; returns ISO date if it looks like a date, else None."""
    if not s:
        return None
    s = s.strip()
    # Try flexible day-mon-year like '21 Aug 2025'
    for fmt in ("%d %b %Y", "%d %B %Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(s, fmt).date().isoformat()
        except Exception:
            pass
    # not a single date; keep original (e.g., year range)
    return None

def _normalize_payload(d):
    """Map incoming keys to your table's columns."""
    out = {
        "title": d.get("title"),
        "label": d.get("label"),
        "level": d.get("level"),
        "description": d.get("description"),
        "years": d.get("years"),
        "availability": d.get("availability"),
        "comments": d.get("comments"),
        "doc_links": d.get("doc_links"),
        "source_type": d.get("source_type"),  # if you created this column
    }
    # Optional dates
    cd = _iso_date_or_none(d.get("change_date"))
    rd = _iso_date_or_none(d.get("reference_date"))
    if cd: out["change_date"] = cd
    if rd: out["reference_date"] = rd
    # leave parent for second pass (we need IDs)
    return {k: v for k, v in out.items() if v not in (None, "")}

def upsert_pages(page_data):
    pass

In [ ]:
_iso_date_or_none('25 Dec 2024')

In [ ]:
_normalize_payload(list(pages.values())[1])

In [ ]:
payload = [_normalize_payload(page) for page in list(pages.values())[:10]]

In [ ]:
payload

In [ ]:
create_record("pages", payload)

In [ ]:
def upsert_pages(page_data):
    # form dict into a list
    payload = [_normalize_payload(page) for page in page_data]
    
    # ensure titles are unique
    # ... TODO
    
    # lookup existing records
    id_map = dict()
    new_records = []
    existing_records = []
    for item in payload:
        record_id = lookup_record_id("pages", "title", item['title'])
        sleep(.5)
        if record_id is not None:
            item["Id"] = record_id
            update_record("pages", item)
        else:
            record_id = create_record("pages", item)["Id"]
        id_map[item["title"]] = record_id
    return id_map

In [ ]:
payload = list(pages.values())

In [ ]:
upsert_pages(payload[:30])

In [ ]:
payload[1]

In [ ]:
create_record("pages", {"title": "x"})